# Bias Demonstration

**How wrong would the estimate be without random assignment?** Companies usually email the customers who look most likely to buy. Comparing emailed with non-emailed customers then mixes the email's effect with the difference between the two groups of customers. This notebook builds such targeted data from the experiment and compares three estimates with the experiment's own result. It answers no challenge question.

**Method:** RCT rejection sampling ([Keith et al., 2023](https://arxiv.org/abs/2307.15176)) drops customers at random, depending on their features and group, so that the email goes mostly to some customers while the experiment's result remains the right answer. The score, the scenarios and the 10% threshold are this project's own choices.

The design was fixed in [`DECISIONS.md`](../DECISIONS.md) (2026-11-25) before this notebook was run:

- **Data:** the Mens E-Mail and No E-Mail groups, 42,613 customers. The benchmark is the experiment's estimate on these customers.
- **Who looks likely to buy:** one point each for a purchase in the last 3 months and for \$200 or more of past-year spend, a score from 0 to 2.
- **Targeting:** in three scenarios, the share of customers emailed for scores 0, 1 and 2 is 50/50/50 (random, a check), 30/50/70 and 10/50/90.
- **Three estimates:** the plain difference; adjusted for both features; adjusted for past-year spend only, as if recency had not been recorded.
- **Criterion:** 1,000 samples per scenario, seed 42. **Bias counts as removed if its absolute value is below 10% of the benchmark.**

**This is a demonstration, not a finding.** Stage 02 had already shown that recent customers spend more even without an email, so the plain difference was expected to overstate the effect.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append("..")
from src.prepare import SEED, load_data

REPS = 1_000
SCENARIOS = {  # share of customers emailed for scores 0, 1 and 2
    "Random (check)": (0.5, 0.5, 0.5),
    "Moderate targeting": (0.3, 0.5, 0.7),
    "Strong targeting": (0.1, 0.5, 0.9),
}
MAX_SHARE = 0.9  # the largest share in any scenario, so every keep probability is at most 1

df = load_data()
df = df[df["segment"] != "Womens E-Mail"].reset_index(drop=True)
emailed = (df["segment"] == "Mens E-Mail").to_numpy()
recent = (df["recency"] <= 3).to_numpy()
high = (df["history"] >= 200).to_numpy()
score = recent.astype(int) + high.astype(int)
spend = df["spend"].to_numpy()

counts = pd.crosstab(pd.Series(score, name="score"), pd.Series(np.where(emailed, "Mens E-Mail", "No E-Mail"), name="group"))
counts["all"] = counts.sum(axis=1)
counts

**Reading the result:** **18% of these customers have a score of 2 and 42% a score of 0,** split about evenly between the two groups, as random assignment implies. These counts use features only.

## Step 1: The experimental benchmark

The difference in mean spend between the Mens E-Mail and No E-Mail groups on all 42,613 customers, as in stage 01.

In [ ]:
benchmark = spend[emailed].mean() - spend[~emailed].mean()
print(f"Benchmark: the experiment's estimate on these {len(df):,} customers, ${benchmark:.4f} per customer")
print(f"Bias counts as removed below ${0.1 * benchmark:.4f} (10% of the benchmark)")

## Step 2: Build targeted samples from the experiment

In each scenario, a customer with score s is kept with probability e ÷ 0.9 if emailed and (1 − e) ÷ 0.9 if not, where e is the scenario's share for that score. Among kept customers with score s, the share emailed is then e.

In [ ]:
def keep_probability(shares):
    """RCT rejection sampling: keep emailed customers with e / 0.9 and the others with (1 - e) / 0.9."""
    e = np.asarray(shares)[score]
    return np.where(emailed, e, 1 - e) / MAX_SHARE


# One sample per scenario, to check the construction. Only assignment and features are used here.
rng = np.random.default_rng(SEED)
checks = {}
for name, shares in SCENARIOS.items():
    kept = rng.random(len(df)) < keep_probability(shares)
    for s in range(3):
        group = score == s
        checks[(name, s)] = {
            "share emailed": emailed[kept & group].mean(),
            "target share": shares[s],
            "share of this score kept": kept[group].mean(),
        }
    checks[(name, "all")] = {"share emailed": emailed[kept].mean(), "target share": np.nan,
                             "share of this score kept": kept.mean()}
pd.DataFrame(checks).T.round(3)

**Reading the result:** **in each scenario, the share emailed within each score matches its target, and every score keeps about 56% of its customers** (0.5 ÷ 0.9). The kept customers therefore have the same mix of scores as all 42,613, so the benchmark still applies, while who received the email now depends on the score. This check uses only assignment and features, not spend.

## Step 3: Three estimates, 1,000 samples per scenario

- **Plain difference:** mean spend of kept emailed customers minus kept non-emailed customers.
- **Adjusted for both features:** the same difference within each combination of recency and past-year spend, weighted by its share of kept customers.
- **Adjusted for past-year spend only:** the same, within the two spend groups only, as if recency had not been recorded.

In [ ]:
both_features = recent.astype(int) * 2 + high.astype(int)  # four combinations
spend_only = high.astype(int)  # two groups


def adjusted(kept, cells):
    """Difference in mean spend within each cell, weighted by the cell's share of kept customers."""
    total = 0.0
    for cell in np.unique(cells):
        inside = kept & (cells == cell)
        total += inside.sum() / kept.sum() * (spend[inside & emailed].mean() - spend[inside & ~emailed].mean())
    return total


ESTIMATORS = {
    "Plain difference": lambda kept: spend[kept & emailed].mean() - spend[kept & ~emailed].mean(),
    "Adjusted for both features": lambda kept: adjusted(kept, both_features),
    "Adjusted for past-year spend only": lambda kept: adjusted(kept, spend_only),
}

rng = np.random.default_rng(SEED)
draws = {}
for name, shares in SCENARIOS.items():
    p = keep_probability(shares)
    values = {estimator: [] for estimator in ESTIMATORS}
    for _ in range(REPS):
        kept = rng.random(len(df)) < p
        for estimator, function in ESTIMATORS.items():
            values[estimator].append(function(kept))
    for estimator, v in values.items():
        draws[(name, estimator)] = np.array(v)

results = pd.DataFrame({
    key: {
        "mean estimate": v.mean(),
        "bias": v.mean() - benchmark,
        "bias as % of benchmark": 100 * (v.mean() - benchmark) / benchmark,
        "empirical standard error": v.std(),
        "Monte Carlo standard error": v.std() / np.sqrt(REPS),
        "2.5%": np.percentile(v, 2.5),
        "97.5%": np.percentile(v, 97.5),
    }
    for key, v in draws.items()
}).T
results.index.names = ["scenario", "estimate"]
results["bias removed"] = np.where(results["bias"].abs() < 0.1 * benchmark, "yes", "no")
results.round(3)

pending

## Figure: each estimate against the experimental benchmark

In [ ]:
BLUE, GRAY, INK = "#2a78d6", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.edgecolor": GRAY,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "text.parse_math": False,
})

fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(11, 3.4), sharex=True, sharey=True)
positions = np.arange(len(ESTIMATORS))
for ax, scenario in zip(axes, SCENARIOS):
    part = results.loc[scenario]
    ax.axvline(benchmark, color=GRAY, linewidth=1, linestyle="--")
    ax.hlines(positions, part["2.5%"], part["97.5%"], color=BLUE, linewidth=2)
    ax.plot(part["mean estimate"], positions, "o", color=BLUE, markersize=8, markeredgecolor="#fcfcfb", markeredgewidth=1.5)
    ax.set_title(scenario, loc="left", fontsize=10, color=INK)
    ax.grid(axis="x", color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0)
axes[0].set_yticks(positions, labels=list(ESTIMATORS))
axes[0].set_ylim(len(ESTIMATORS) - 0.5, -0.7)
axes[1].set_xlabel("Estimated extra spend per customer from the Mens email (USD)")
fig.suptitle("TITLE", x=0.01, ha="left", fontsize=12)
fig.text(0.01, 0.875, f"Dots: mean of {REPS:,} samples; lines: middle 95% of samples; dashed: the experiment's result (${benchmark:.2f})",
         ha="left", fontsize=9, color=INK)
plt.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig("../reports/figures/bias_demonstration.png", dpi=150, bbox_inches="tight")
plt.show()

pending